# Graphql

> Fill in a module description here

In [ ]:
# | default_exp graphql

In [ ]:
# | hide
from nbdev.showdoc import *  # type: ignore

In [2]:
import os
import subprocess
import sys
from dotenv import load_dotenv

if os.getenv("USER") == "max":
    # Get the current working directory
    current_dir = os.getcwd()

    # Append the parent directory to sys.path
    parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
    sys.path.append(parent_dir)

    # Decrypt the .env.local file and load variables
    try:
        # Decrypt and output to stdout
        os.system(
            f"dotenvx decrypt -f {os.path.abspath(os.path.join(current_dir, '..'))}/.env.local"
        )

        # Load the decrypted environment variables from stdout
        load_dotenv(
            dotenv_path=os.path.abspath(os.path.join(current_dir, "..")) + "/.env.local"
        )

        os.system(
            f"dotenvx encrypt -f {os.path.abspath(os.path.join(current_dir, '..'))}/.env.local"
        )

        print("Environment variables loaded successfully.")
        os.environ["DOCKER_HOST"] = "unix:///Users/max/.docker/run/docker.sock"
    except subprocess.CalledProcessError as e:
        print(f"Error decrypting .env.local: {e}")
        sys.exit(1)

✔ decrypted (/Users/max/Documents/product_horse/.env.local)
✔ encrypted (/Users/max/Documents/product_horse/.env.local)
Environment variables loaded successfully.


In [3]:
# | export
from typing import (
    Annotated,
    Any,
    Union,
    cast,
    Sequence,
    Callable,
    TypeVar,
    TypedDict,
    List,
    Optional,
    Tuple,
)
from functools import wraps
from enum import Enum
from uuid import UUID
import strawberry
import jwt
from jwt.exceptions import InvalidTokenError
from fastapi import FastAPI
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware
from strawberry.extensions import AddValidationRules
from graphql.validation import NoSchemaIntrospectionCustomRule
from product_horse.filesystems import LocalFileSystem
from product_horse.db import (
    SqlModelDatabase,
    Employee,
    PermissionLevel as DbPermissionLevel,
    Company,
    UnvalidatedCompany,
    CreateEmployee,
    User as DbUser,
    Utterance,
    UnvalidatedUser,
    Word,
    Video,
    Clip,
    FileMetadata,
    UnvalidatedFileMetadata,
    UnvalidatedTranscription,
    UnvalidatedUtterance,
    UnvalidatedVideo,
    RenderStatus,
    FileType as DbFileType,
)
from datetime import datetime, timedelta, timezone

import aiohttp
from pydantic import ValidationError
from product_horse.audio import (
    AudioEditor,
)
from product_horse.api import get_relevant_utterances_from_query

from strawberry.permission import BasePermission
from strawberry.file_uploads import Upload
from strawberry.fastapi import BaseContext, GraphQLRouter
from starlette.datastructures import UploadFile
from tenacity import retry, stop_after_attempt, wait_exponential

from dotenv import load_dotenv
import asyncio
import os
import logging
import boto3
from botocore.config import Config
from fastapi.staticfiles import StaticFiles
from fastapi.responses import FileResponse

if not os.getenv("DATABASE_URL"):
    load_dotenv()

# Auth and context

In [ ]:
# |export
API_URL = "https://storage.producthorse.workers.dev/"  # or localhost:8787 - but won't work with transcription
# API_URL = "http://localhost:8787"

logger = logging.getLogger(__name__)

In [ ]:
# | export
database_url = os.getenv("DATABASE_URL")
database_superuser_url = os.getenv("DATABASE_SUPERUSER_URL")
if database_url is None or database_superuser_url is None:
    raise ValueError("DATABASE_URL or DATABASE_SUPERUSER_URL is not set")
secret = os.getenv("SECRET")
if secret is None:
    raise ValueError("SECRET is not set")
database = SqlModelDatabase(database_url=database_url)
database_superuser = SqlModelDatabase(database_url=database_superuser_url)

local_file_system = LocalFileSystem(base_path="")

config = Config(
    region_name="auto",
    signature_version="s3v4",
)

s3_client = boto3.client("s3", config=config)

MB_100 = 100 * 1024 * 1024
PART_SIZE = 5 * 1024 * 1024  # 5MB part size


class JwtPayload(TypedDict):
    id: UUID
    company_id: UUID
    exp: float
    permission_level: DbPermissionLevel


def create_jwt(
    employee_id: UUID, company_id: UUID, permission_level: DbPermissionLevel
) -> str:
    expiration = datetime.now(timezone.utc) + timedelta(weeks=2)
    payload = {
        "id": str(employee_id),
        "company_id": str(company_id),
        "exp": expiration,
        "permission_level": permission_level.value,
    }
    return jwt.encode(payload, secret, algorithm="HS256")  # type: ignore


def decode_jwt(token: str) -> JwtPayload:
    payload = cast(JwtPayload, jwt.decode(token, secret, algorithms=["HS256"]))  # type: ignore
    if "id" not in payload:
        raise InvalidTokenError
    if "company_id" not in payload:
        raise InvalidTokenError
    exp_timestamp = datetime.fromtimestamp(payload["exp"], tz=timezone.utc)
    if exp_timestamp < datetime.now(timezone.utc):
        raise InvalidTokenError
    return payload


class Context(BaseContext):
    @property
    def employee(self) -> Optional[Employee]:
        request = self.request
        if request is None:
            print("Request is None")
            return None
        authorization = request.headers.get("Authorization")
        if authorization is None:
            return None
        if not authorization.startswith("Bearer "):
            return None
        token = authorization.split()[1]
        payload = decode_jwt(token)
        employee_id = payload["id"]
        # run on superuser connection, since it's behind jwt auth
        employee = database_superuser.get_employee(employee_id=employee_id)
        return employee

    @property
    def jwt(self) -> Optional[str]:
        request = self.request
        if request is None:
            return None
        authorization = request.headers.get("Authorization")
        if authorization is None:
            return None
        if not authorization.startswith("Bearer "):
            return None
        token = authorization.split()[1]
        return token


class IsAuthenticated(BasePermission):
    message = "Employee is not authenticated"
    error_extensions = {"code": "UNAUTHORIZED"}

    def has_permission(self, source: Any, info: strawberry.Info, **kwargs: Any) -> bool:
        employee = info.context.employee
        return employee is not None


T = TypeVar("T")


@strawberry.type
class FormError:
    field: str
    message: str


@strawberry.type
class FormErrors:
    errors: list[FormError]


def handle_form_validation_errors(
    func: Callable[..., T],
) -> Callable[..., Union[T, FormErrors]]:
    @wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> Union[T, FormErrors]:
        try:
            return func(*args, **kwargs)
        except ValidationError as e:
            errors = [
                FormError(field=cast(str, error["loc"][-1]), message=error["msg"])
                for error in e.errors()
            ]
            return FormErrors(errors=errors)

    return wrapper

# Helpers

In [ ]:
# | export


from product_horse.db import Transcription


def create_presigned_url(
    object_name: str,
    expiration: int = 3600,
    method: str = "get_object",
) -> Optional[str]:
    """Generate a presigned URL to share an S3 object

    :param bucket_name: string
    :param object_name: string
    :param expiration: Time in seconds for the presigned URL to remain valid
    :return: Presigned URL as string. If error, returns None.
    """

    bucket_name = os.environ["FLY_BUCKET_NAME"]
    if not bucket_name:
        raise ValueError("FLY_BUCKET_NAME environment variable not found")

    try:
        response = cast(
            str,
            s3_client.generate_presigned_url(  # type: ignore
                method,
                Params={
                    "Bucket": bucket_name,
                    "Key": object_name,
                },
                ExpiresIn=expiration,
            ),
        )
    except Exception as e:
        logging.error(e)
        return None

    return response


api_key = os.getenv("ASSEMBLYAI_API_KEY")
if not api_key:
    raise ValueError("ASSEMBLYAI_API_KEY environment variable not found")
audio_editor = AudioEditor(api_key=api_key)


@retry(stop=stop_after_attempt(2), wait=wait_exponential(multiplier=1, min=4, max=15))
async def transcribe_and_save_file(
    file_metadata: FileMetadata,
    signed_url: str,
    employee: Employee,
    db: SqlModelDatabase,
) -> Transcription:
    # check that employee has access to the file
    if not hasattr(file_metadata, "id"):
        raise Exception("File metadata does not have an id")
    file_metadata_to_verify = db.as_employee(employee).get_file_metadata(
        file_metadata.id
    )
    if file_metadata_to_verify is None:
        raise Exception("File metadata not found")
    transcription_result = await audio_editor.transcribe(signed_url, employee)
    utterances: List[UnvalidatedUtterance] = transcription_result.utterances
    unvalidated_transcription = UnvalidatedTranscription(
        file_id=file_metadata.id,
        text=transcription_result.text,
        company_id=employee.company_id,
        created_by_id=employee.id,
    )
    transcription: Transcription = db.as_employee(employee).save_transcription(
        unvalidated_transcription, utterances
    )
    return transcription


async def run_remote_transcribe_and_save_file(
    file_metadata: FileMetadata,
    employee: Employee,
) -> Transcription:
    metadata = database.as_employee(employee).get_file_metadata(file_metadata.id)
    if metadata is None:
        raise Exception("File metadata not found")
    signed_url = create_presigned_url(metadata.file_path)
    if signed_url is None:
        raise Exception("Failed to create presigned URL")
    print("transcribe_and_save_file")
    return await transcribe_and_save_file(
        db=database,
        file_metadata=file_metadata,
        signed_url=signed_url,
        employee=employee,
    )

# Graphql Types

In [ ]:
# | export
PermissionsLevel = strawberry.enum(DbPermissionLevel)


@strawberry.experimental.pydantic.type(Employee)
class Employee:
    id: UUID
    email: str
    name: str
    permission_level: PermissionsLevel  # type: ignore


@strawberry.experimental.pydantic.type(DbUser)
class User:
    id: UUID
    name: str


@strawberry.experimental.pydantic.type(Word)
class Word:
    id: UUID
    text: str
    start: float
    end: float


@strawberry.input
class UtteranceSegmentInput:
    utterance_id: str
    word_ids: list[str]


@strawberry.experimental.pydantic.type(Utterance)
class Utterance:
    id: UUID
    confidence: float
    start_time: float
    end_time: float
    speaker: str
    text: str
    words: list[Word]


@strawberry.experimental.pydantic.type(Clip)
class Clip:
    id: UUID
    name: str
    words: list[Word]


@strawberry.experimental.pydantic.type(Video)
class Video:
    id: UUID
    title: str
    clips: list[Clip]
    title: str
    file_path: str
    resolution_x: int
    resolution_y: int
    render_status: str
    fps: int
    signed_url: Optional[str] = None
    visibility: str


@strawberry.experimental.pydantic.type(Company)
class Company:
    id: UUID
    name: str
    employees: list[Employee]


@strawberry.enum
class FileStatus(str, Enum):
    upload_started = "upload_started"
    upload_finished = "upload_finished"
    upload_failed = "upload_failed"
    transcribe_started = "transcribe_started"
    transcribe_finished = "transcribe_finished"
    transcribe_failed = "transcribe_failed"
    active = "active"
    deleted = "deleted"
    archived = "archived"


@strawberry.experimental.pydantic.type(FileMetadata)
class FileMetadata:
    id: UUID
    file_name: str
    file_path: str
    file_status: FileStatus


@strawberry.experimental.pydantic.type(Transcription)
class Transcription:
    id: UUID
    text: str
    created_at: datetime
    updated_at: datetime
    embedded: Optional[bool]
    utterances: list[Utterance]
    file_metadata: FileMetadata


@strawberry.type
class RegisterCompanySuccess:
    company: Company
    token: str


@strawberry.enum
class FileType(Enum):
    VIDEO = "video"
    AUDIO = "audio"
    TEXT = "text"


@strawberry.input
class FileMetadataInput:
    id: Optional[str] = None
    file_name: str
    file_type: FileType
    file_size: int
    resolution_x: int
    resolution_y: int
    frame_rate: int
    duration: float
    mime_type: Optional[str] = None


@strawberry.type
class MultiPartUploadUrls:
    type: str
    presigned_urls: list[str]
    complete_post_url: str
    parts: int


@strawberry.type
class SinglePartUpload:
    type: str
    url: str


@strawberry.type
class UploadConfig:
    type: str
    destination: Union[MultiPartUploadUrls, SinglePartUpload]


@strawberry.type
class FileMetadataWithSignedUrl:
    id: UUID
    file_name: str
    file_path: str
    file_status: FileStatus
    upload_config: Optional[UploadConfig] = None


# Create a Union type to represent the 2 results from the mutation
RegisterResponse = Annotated[
    Union[RegisterCompanySuccess, FormErrors],
    strawberry.union("RegisterCompanyResponse"),
]


@strawberry.type
class LoginSuccess:
    employee: Employee
    token: str


LoginResponse = Annotated[
    Union[LoginSuccess, FormErrors],
    strawberry.union("LoginResponse"),
]

# External calls


In [ ]:
# | export

from typing import Any


class CreateVideoRequest(BaseModel):
    video_id: str
    utterance_segments_inputs: List[UtteranceSegmentInput]
    employee_id: str
    jwt: str
    title: str
    size: Tuple[int, int] = (1920, 1080)
    force_size: bool = True


async def create_video_async(
    api_url: str,
    jwt_token: str,
    video_id: str,
    utterance_segments_inputs: List[UtteranceSegmentInput],
    employee_id: str,
    title: str,
    size: Tuple[int, int] = (1920, 1080),
    force_size: bool = True,
) -> None:
    headers = {
        "Authorization": f"Bearer {jwt_token}",
        "Content-Type": "application/json",
    }
    logger.info(f"Sending request to {api_url}/create_video")

    request_data = CreateVideoRequest(
        video_id=str(video_id),
        utterance_segments_inputs=utterance_segments_inputs,
        employee_id=str(employee_id),
        jwt=jwt_token,
        title=title,
        size=size,
        force_size=force_size,
    )
    request_data_json = request_data.model_dump()

    async with aiohttp.ClientSession() as session:
        try:
            async with session.post(
                f"{api_url}/create_video", json=request_data_json, headers=headers
            ) as response:
                logger.info(f"Response: {response}")
                if 200 <= response.status <= 299:
                    logger.info("Video creation request sent successfully")
                else:
                    logger.error(f"Error: Received status code {response.status}")
                    response_text = await response.text()
                    employee = database_superuser.get_employee(employee_id)
                    if employee is None:
                        raise Exception("Employee not found")
                    video = database.as_employee(employee).get_video(video_id)  # type: ignore
                    if video is None:
                        raise Exception("Video not found")
                    video.render_status = RenderStatus.failed
                    database.as_employee(employee).save_video(video)  # type: ignore
                    logger.error(f"Response body: {response_text}")
        except aiohttp.ClientError as e:
            logger.error(f"Error sending request: {str(e)}")


def create_multipart_upload_urls(
    object_name: str,
    file_size: int,
    expiration: int = 3600,
    bucket: str = "product-horse-storage",
) -> MultiPartUploadUrls:
    # Create multipart upload
    response = s3_client.create_multipart_upload(Bucket=bucket, Key=object_name)  # type: ignore
    upload_id: str = response["UploadId"]  # type: ignore

    # Calculate number of parts
    num_parts = (file_size + PART_SIZE - 1) // PART_SIZE

    # Generate presigned URLs for each part
    presigned_urls = []
    for part_number in range(1, num_parts + 1):
        url: str = s3_client.generate_presigned_url(  # type: ignore
            "upload_part",
            Params={
                "Bucket": bucket,
                "Key": object_name,
                "UploadId": upload_id,
                "PartNumber": part_number,
            },
            ExpiresIn=expiration,
        )
        presigned_urls.append(url)  # type: ignore

    # Generate presigned URL for completing the multipart upload
    complete_url: str = s3_client.generate_presigned_url(  # type: ignore
        "complete_multipart_upload",
        Params={"Bucket": bucket, "Key": object_name, "UploadId": upload_id},
        ExpiresIn=expiration,
    )

    return MultiPartUploadUrls(
        upload_id=upload_id,  # type: ignore
        presigned_urls=presigned_urls,  # type: ignore
        complete_post_url=complete_url,  # type: ignore
        parts=num_parts,
    )

# Queries

In [ ]:
# | export
@strawberry.type
class Query:
    employee: Employee

    @strawberry.field(permission_classes=[IsAuthenticated])
    def get_users(self, info: strawberry.Info) -> Sequence[User]:
        return database.as_employee(info.context.employee).get_all_users()  # type: ignore

    @strawberry.field(permission_classes=[IsAuthenticated])
    def get_transcripts(self, info: strawberry.Info) -> Sequence[Transcription]:
        transcripts = database.as_employee(
            info.context.employee
        ).get_all_unique_transcriptions(mode="file_name")
        return transcripts  # type: ignore

    @strawberry.field(permission_classes=[IsAuthenticated])
    async def get_relevant_utterances(
        self, info: strawberry.Info, query: str, transcript_ids: Sequence[str]
    ) -> Sequence[Utterance]:
        transcript_uuids = [UUID(transcript_id) for transcript_id in transcript_ids]
        transcripts = database.as_employee(info.context.employee).get_transcriptions(
            transcription_ids=transcript_uuids
        )
        # TODO: this won't work if there are too few utterances to query. Add a check.
        # TODO: check the seconds buffer and N to return in the search_engine. Might be worth exposing those,
        # and I am not sure what seconds buffer is anyway.
        utterances = await get_relevant_utterances_from_query(
            db=database,
            employee=info.context.employee,
            query=query,
            transcripts=transcripts,
        )
        return utterances  # type: ignore

    @strawberry.field(permission_classes=[IsAuthenticated])
    def get_video(self, info: strawberry.Info, video_id: str) -> Video:
        video = database.as_employee(info.context.employee).get_video(video_id)
        if video is None:
            raise Exception("Video not found")
        if video.file_path is None:
            raise Exception("Video file path is None")
        signed_url = create_presigned_url(video.file_path)
        if signed_url is None:
            raise Exception("Failed to create presigned URL")
        video_dict = video.model_dump()
        # Remove unwanted keys from the dictionary
        keys_to_remove = ["created_by_id", "company_id", "updated_at", "created_at"]
        for key in keys_to_remove:
            video_dict.pop(key, None)
        video_dict["signed_url"] = signed_url
        try:
            if video_dict["clips"] is None:
                video_dict["clips"] = video.clips or []
        except KeyError as e:
            print(e)
            video_dict["clips"] = []
        return Video(**video_dict)

    @strawberry.field(permission_classes=[IsAuthenticated])
    def get_all_videos(self, info: strawberry.Info) -> Sequence[Video]:
        return database.as_employee(info.context.employee).get_all_videos()  # type: ignore

# Mutations

In [ ]:
# | export
@strawberry.type
class Mutation:
    """
    Mutations are the functions that change the state of the database.
    """

    @strawberry.mutation
    def login(self, email: str, password: str) -> LoginResponse:
        employee = database_superuser.authenticate_employee(email, password)
        if employee is None:
            raise Exception("Invalid email or password")
        return LoginSuccess(
            employee=employee,  # type: ignore
            token=create_jwt(
                employee.id, employee.company_id, employee.permission_level
            ),
        )

    @strawberry.mutation
    @handle_form_validation_errors
    def register_company_and_employee(
        self, name: str, email: str, password: str, company_name: str
    ) -> RegisterResponse:
        company_to_save = UnvalidatedCompany(name=company_name)
        employee_to_save = CreateEmployee(
            name=name,
            email=email,
            password=password,
            permission_level=DbPermissionLevel.ADMIN,
        )
        company, employee = database_superuser.save_company_and_employee(
            company_to_save, employee_to_save
        )
        return RegisterCompanySuccess(
            company=company,  # type: ignore
            token=create_jwt(employee.id, company.id, employee.permission_level),
        )

    @strawberry.mutation(permission_classes=[IsAuthenticated])
    def update_file_metadata_status(
        self,
        info: strawberry.Info[Context, Any],
        file_id: str,
        file_status: FileStatus,
    ) -> FileMetadata:
        return database.as_employee(info.context.employee).update_file_metadata(
            file_id, {"file_status": file_status}
        )  # type: ignore

    @strawberry.mutation(permission_classes=[IsAuthenticated])
    def save_user(
        self,
        info: strawberry.Info[Context, Any],
        name: str,
        external_id: Optional[str] = None,
    ) -> User:
        external_id = external_id if external_id is not None else ""
        if info.context.employee is None:
            raise Exception("Employee is None")
        user = UnvalidatedUser(
            name=name,
            external_id=external_id,
            company_id=info.context.employee.company_id,
            created_by_id=info.context.employee.id,
        )
        return database.as_employee(info.context.employee).save_user(user)  # type: ignore

    @strawberry.mutation(permission_classes=[IsAuthenticated])
    async def transcribe_file(
        self,
        info: strawberry.Info,
        file_id: str,
    ) -> bool:
        file_metadata = database.as_employee(info.context.employee).get_file_metadata(
            file_id
        )
        if file_metadata is None:
            raise Exception("File metadata is None")
        await run_remote_transcribe_and_save_file(
            file_metadata=file_metadata,
            employee=info.context.employee,
        )
        return True

    @strawberry.mutation(permission_classes=[IsAuthenticated])
    async def save_files(
        self,
        info: strawberry.Info,
        user_id: UUID,
        file_metadata: List[FileMetadataInput],
    ) -> List[FileMetadataWithSignedUrl]:
        user = database.as_employee(info.context.employee).get_user(user_id)
        if user is None:
            raise Exception("User not found")
        metadata: List[FileMetadataWithSignedUrl] = []
        for file in file_metadata:
            try:
                if file.file_name is None:  # type: ignore
                    raise Exception("File name is None")
                # Generate a unique file key
                unique_file_key = await local_file_system.get_unique_file_key(
                    file.file_name, str(user.id)
                )
                original_file_name = unique_file_key.split("__")[1]
                unique_file_key = unique_file_key.strip("/")  # remove leading /
                unvalidated_metadata = UnvalidatedFileMetadata(
                    user_id=user.id,
                    file_name=original_file_name,
                    file_path=unique_file_key,
                    file_type=DbFileType[file.file_type.value],
                    created_by_id=info.context.employee.id,
                    company_id=info.context.employee.company_id,
                    resolution_x=file.resolution_x,
                    resolution_y=file.resolution_y,
                    frame_rate=file.frame_rate,
                )
                file_metadata_to_save = database.as_employee(
                    info.context.employee
                ).save_file_metadata(unvalidated_metadata)
                signed_urls = None
                signed_url = None
                file_type = "single"
                if file.file_size > MB_100:
                    file_type = "multi"
                    signed_urls = create_multipart_upload_urls(
                        file_metadata_to_save.file_path, file.file_size
                    )
                else:
                    signed_url = create_presigned_url(
                        file_metadata_to_save.file_path,
                        method="put_object",
                    )
                if signed_url is None and signed_urls is None:
                    raise Exception("Failed to create presigned URL")
                destination = (
                    signed_urls
                    if file_type == "multi"
                    else SinglePartUpload(type=file_type, url=signed_url) # type: ignore
                )
                if destination is None:
                    print("destination is None")
                    raise Exception("Failed to create presigned URL")
                file_metadata_to_save_with_signed_url = FileMetadataWithSignedUrl(
                    id=file_metadata_to_save.id,
                    file_name=file_metadata_to_save.file_name,
                    file_path=file_metadata_to_save.file_path,
                    file_status=file_metadata_to_save.file_status,  # type: ignore
                    upload_config=UploadConfig(
                        type=file_type,
                        destination=destination,
                    ),
                )
                metadata.append(file_metadata_to_save_with_signed_url)  # type: ignore
            except Exception as e:
                print(e)
                raise Exception(f"Failed to upload file {file}")
        return metadata

    @strawberry.mutation(permission_classes=[IsAuthenticated])
    async def create_video_from_utterances(
        self,
        info: strawberry.Info,
        utterance_segments: Sequence[UtteranceSegmentInput],
        title: str,
    ) -> Video:
        """TODO
        Refactor to be an initializer and then you query a different endpoint to get the video.
        Use render status appropriately.
        Mirror the v2 file upload design for this endpoint.
        1. [DONE] create_video_from_utterances is an initializer, send id and respond with a video created
        - in background, kick off serverless video creation on modal.com. Use python storage_client to get the files there, not here.
        -  service talk directly to the database and update the video with the status as it goes.
        2. get_video, responds with video and status, along with a signed url to download the video if it's ready.
        - signed url is for the r2 api, not the graphql server, so the client can download the video directly.
        3. Add more granularity to the video and file status - upload created, upload completed, transcription in progress, etc.
        """
        size = (int(1920 / 4), int(1080 / 4))
        video_to_make = UnvalidatedVideo(
            title=title,
            company_id=info.context.employee.company_id,
            created_by_id=info.context.employee.id,
            file_path="",
        )
        video_shell = database.as_employee(info.context.employee).save_video(
            video_to_make
        )
        asyncio.create_task(
            create_video_async(
                api_url="http://gpu-product-horse.flycast",
                jwt_token=info.context.jwt,
                video_id=str(video_shell.id),
                utterance_segments_inputs=utterance_segments,  # type: ignore
                employee_id=info.context.employee.id,
                title=title,
                size=size,
                force_size=True,
            )
        )
        return video_shell  # type: ignore

In [1]:
# | export
async def get_context() -> Context:
    return Context()


schema = strawberry.Schema(
    Query,
    Mutation,
    scalar_overrides={UploadFile: Upload},
    extensions=[
        AddValidationRules([NoSchemaIntrospectionCustomRule]),
    ],
)

graphql_app = GraphQLRouter(  # type: ignore
    schema,
    context_getter=get_context,
    graphql_ide=None,
)

app = FastAPI()

origins = [
    "http://127.0.0.1:8000",
    "http://127.0.0.1:3000",
    "http://127.0.0.1:5173",
    "https://product-horse.fly.dev",
]

app.mount("/assets", StaticFiles(directory="./frontend/dist/assets"), name="assets")

app.include_router(graphql_app, prefix="/graphql")  # type: ignore


@app.get("/{full_path:path}")
async def serve_react_app(full_path: str):
    print(full_path)
    return FileResponse("./frontend/dist/index.html")


app.add_middleware(
    CORSMiddleware,
    allow_origins=origins,
    allow_credentials=True,
    allow_methods=["GET", "POST", "OPTIONS"],
    allow_headers=["*"],
)


NameError: name 'Context' is not defined

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()  # type: ignore